# Chapter 4: Regression and Model Selection

## Data-Driven Science and Engineering — Review Notebook

This notebook combines concise review notes with modernized Python versions of the nine official Chapter 4 companion examples.

The chapter develops a single central idea: fitting the observed data is not enough. A useful model must also control complexity and generalize to unseen data.

The worked examples cover:

1. Linear regression with $L_\infty$, $L_1$, and $L_2$ losses
2. Gradient descent on convex and nonconvex objectives
3. Overdetermined and underdetermined systems
4. Comparisons among classical and regularized regression methods
5. Pareto fronts and parsimonious models
6. Holdout and $K$-fold cross-validation
7. KL divergence, AIC, and BIC for model validation

### References

- Brunton and Kutz, *Data-Driven Science and Engineering*, Chapter 4
- [Official Chapter 4 outline](https://databookuw.com/page/page-7/)
- [Official Python companion repository](https://github.com/dynamicslab/databook_python/tree/master/CH04)

The explanations and implementations below are newly organized and annotated. They preserve the mathematical purpose of the official demonstrations while replacing outdated library calls and adding reproducibility and validation checks.

# Chapter Overview

Given observations $(x_i,y_i)$, regression selects parameters $\beta$ for a model

$$
\hat y_i=f(x_i;\beta).
$$

Training is usually expressed as an optimization problem:

$$
\hat\beta=\arg\min_\beta L(\beta),
$$

where the **loss function** measures disagreement between predictions and observations.

For squared-error regression,

$$
L(\beta)
=
\frac{1}{n}\sum_{i=1}^{n}
\left(y_i-f(x_i;\beta)\right)^2.
$$

In matrix form,

$$
L(\beta)=\|y-X\beta\|_2^2.
$$

## Regularization

Minimizing training loss alone can produce an unnecessarily complicated model. Regularization adds a complexity penalty:

$$
\hat\beta
=
\arg\min_\beta
\left[L(\beta)+\lambda R(\beta)\right].
$$

- $L(\beta)$ measures data-fitting error.
- $R(\beta)$ measures undesirable complexity.
- $\lambda$ controls the fit-versus-simplicity tradeoff.

Common choices include

$$
R(\beta)=\|\beta\|_2^2 \quad \text{(ridge)},
$$

$$
R(\beta)=\|\beta\|_1 \quad \text{(LASSO)},
$$

and

$$
R(\beta)=\|\beta\|_0 \quad \text{(subset selection)}.
$$

The complete quantity $L(\beta)+\lambda R(\beta)$ is the **objective function**.

## Cross-validation

Training error normally falls as model complexity increases, so it cannot determine which model will generalize best. Cross-validation estimates performance on observations excluded from fitting.

For $K$ folds,

$$
CV(\lambda)
=
\frac{1}{K}\sum_{k=1}^{K}
L_{\text{validation},k}(\lambda).
$$

The selected hyperparameter is

$$
\lambda^*=\arg\min_\lambda CV(\lambda).
$$

The test set remains untouched until the model and its hyperparameters have been selected.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import linprog, minimize, minimize_scalar
from scipy.stats import norm

from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet, HuberRegressor, Lasso, LassoCV, Ridge, RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)

SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.25,
})


def polynomial_matrix(x, degree):
    '''Return columns [1, x, x^2, ..., x^degree].'''
    x = np.asarray(x)
    return np.column_stack([x**power for power in range(degree + 1)])


def rmse(y_true, y_pred):
    '''Root-mean-square error.'''
    return np.sqrt(mean_squared_error(y_true, y_pred))

# 4.1 Classical Curve Fitting and Least-Squares Regression

For a linear model,

$$
\hat y=\beta_1x+\beta_0,
$$

the residuals are

$$
r_i=\hat y_i-y_i.
$$

Different residual norms produce different regression behavior:

$$
E_\infty=\max_i|r_i|,
\qquad
E_1=\sum_i|r_i|,
\qquad
E_2=\sum_i r_i^2.
$$

- $L_\infty$ minimizes the single largest error.
- $L_1$ minimizes total absolute error and is relatively robust to outliers.
- $L_2$ is ordinary least squares and places extra weight on large errors.

The first official example compares these three fits with and without an outlier.

In [ ]:
def line_loss(parameters, x, y, loss_type):
    slope, intercept = parameters
    residuals = slope * x + intercept - y

    if loss_type == "L_inf":
        return np.max(np.abs(residuals))
    if loss_type == "L1":
        return np.sum(np.abs(residuals))
    if loss_type == "L2":
        return np.sum(residuals**2)
    raise ValueError("loss_type must be 'L_inf', 'L1', or 'L2'")


def fit_line_by_loss(x, y, loss_type):
    result = minimize(
        line_loss,
        x0=np.array([0.0, np.mean(y)]),
        args=(x, y, loss_type),
        method="Nelder-Mead",
        options={"maxiter": 20_000, "xatol": 1e-10, "fatol": 1e-10},
    )
    if not result.success:
        raise RuntimeError(result.message)
    return result.x


x_loss = np.arange(1, 11, dtype=float)
y_clean = np.array([0.2, 0.5, 0.3, 0.7, 1.0, 1.5, 1.8, 2.0, 2.3, 2.2])
y_outlier = y_clean.copy()
y_outlier[3] = 3.5

loss_types = ["L_inf", "L1", "L2"]
fit_results = {}

for data_name, y_values in {"Clean data": y_clean, "With outlier": y_outlier}.items():
    for loss_type in loss_types:
        fit_results[(data_name, loss_type)] = fit_line_by_loss(
            x_loss, y_values, loss_type
        )

coefficient_rows = []
for (data_name, loss_type), coefficients in fit_results.items():
    coefficient_rows.append({
        "Dataset": data_name,
        "Loss": loss_type,
        "Slope": coefficients[0],
        "Intercept": coefficients[1],
    })

coefficient_table = pd.DataFrame(coefficient_rows)
print(coefficient_table.round(4).to_string(index=False))

In [ ]:
x_fit = np.linspace(0, 11, 300)
colors = {"L_inf": "tab:green", "L1": "tab:orange", "L2": "tab:blue"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (data_name, y_values) in zip(
    axes, {"Clean data": y_clean, "With outlier": y_outlier}.items()
):
    ax.scatter(x_loss, y_values, color="black", s=45, label="Observed data", zorder=3)

    for loss_type in loss_types:
        slope, intercept = fit_results[(data_name, loss_type)]
        ax.plot(
            x_fit,
            slope * x_fit + intercept,
            linewidth=2.2,
            color=colors[loss_type],
            label=loss_type,
        )

    ax.set_title(data_name)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_ylim(-0.2, 4.1)
    ax.legend()

fig.suptitle("Linear Fits Under Different Loss Functions", y=1.02)
plt.tight_layout()
plt.show()

## Polynomial least squares

A polynomial model

$$
\hat y=\beta_0+\beta_1x+\cdots+\beta_dx^d
$$

is nonlinear in $x$ but linear in its unknown coefficients. Constructing a polynomial feature matrix converts the problem into linear least squares:

$$
\hat\beta=\arg\min_\beta\|y-X\beta\|_2^2.
$$

`np.linalg.lstsq` solves this problem without explicitly forming $(X^TX)^{-1}$.

In [ ]:
rng_poly = np.random.default_rng(SEED)
x_poly = np.linspace(-1, 1, 60)
y_poly_true = 1.0 - 1.5 * x_poly + 0.8 * x_poly**2
y_poly_observed = y_poly_true + 0.18 * rng_poly.standard_normal(x_poly.size)

poly_degree = 3
X_poly = polynomial_matrix(x_poly, poly_degree)
beta_poly = np.linalg.lstsq(X_poly, y_poly_observed, rcond=None)[0]
y_poly_fit = X_poly @ beta_poly

print("Estimated coefficients:", np.round(beta_poly, 4))
print(f"Training RMSE: {rmse(y_poly_observed, y_poly_fit):.4f}")

plt.figure(figsize=(10, 5))
plt.scatter(x_poly, y_poly_observed, color="gray", alpha=0.7, label="Noisy observations")
plt.plot(x_poly, y_poly_true, color="black", linewidth=2.5, label="Underlying function")
plt.plot(x_poly, y_poly_fit, color="tab:red", linewidth=2.2, label="Degree-3 least-squares fit")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Polynomial Least-Squares Regression")
plt.legend()
plt.show()

# 4.2 Nonlinear Regression and Gradient Descent

When a model depends nonlinearly on its parameters, a direct least-squares formula may not exist. Gradient descent updates the parameters iteratively:

$$
\beta^{(k+1)}
=
\beta^{(k)}-\eta_k\nabla J\left(\beta^{(k)}\right).
$$

The negative gradient gives the direction of steepest local decrease. The learning rate $\eta_k$ determines the step size.

For a convex quadratic, gradient descent approaches the unique global minimum. For a nonconvex surface, different starting points may lead to different local minima.

In [ ]:
H_quadratic = np.diag([2.0, 6.0])


def quadratic_value(point):
    x_value, y_value = point
    return x_value**2 + 3.0 * y_value**2


def quadratic_gradient(point):
    return H_quadratic @ point


def quadratic_gradient_descent(start, tolerance=1e-10, max_iterations=100):
    point = np.asarray(start, dtype=float)
    history = [point.copy()]

    for _ in range(max_iterations):
        gradient = quadratic_gradient(point)
        if np.linalg.norm(gradient) < tolerance:
            break

        # Exact line-search step for a positive-definite quadratic.
        step_size = (gradient @ gradient) / (gradient @ H_quadratic @ gradient)
        point = point - step_size * gradient
        history.append(point.copy())

    return np.array(history)


quadratic_path = quadratic_gradient_descent(start=[3.0, 2.0])
quadratic_objective = np.array([quadratic_value(point) for point in quadratic_path])

print("Iterations:", len(quadratic_path) - 1)
print("Final point:", np.round(quadratic_path[-1], 10))
print("Final objective:", quadratic_objective[-1])

In [ ]:
grid = np.linspace(-4, 4, 300)
X_quad, Y_quad = np.meshgrid(grid, grid)
Z_quad = X_quad**2 + 3 * Y_quad**2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].contour(X_quad, Y_quad, Z_quad, levels=20, cmap="viridis")
axes[0].plot(quadratic_path[:, 0], quadratic_path[:, 1], "o-", color="crimson")
axes[0].scatter(0, 0, marker="*", s=180, color="black", label="Global minimum")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_title("Gradient-Descent Path")
axes[0].legend()

axes[1].semilogy(quadratic_objective, "o-", color="tab:blue")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel(r"Objective $f(x,y)$")
axes[1].set_title("Objective Convergence")

plt.tight_layout()
plt.show()

## Nonconvex and alternating descent

A nonconvex objective can contain several basins of attraction. Gradient descent only uses local slope information, so initialization matters.

**Alternating descent**, also called coordinate descent, minimizes one coordinate at a time while holding the others fixed. It often produces a characteristic axis-aligned path.

In [ ]:
def nonconvex_surface(point):
    x_value, y_value = np.asarray(point)
    q1 = 3 * (x_value + 3)**2 + (y_value + 3)**2
    q2 = 3 * (x_value - 3)**2 + (y_value - 3)**2
    return 2.0 - 1.6 * np.exp(-0.05 * q1) - np.exp(-0.1 * q2)


def nonconvex_gradient(point):
    x_value, y_value = np.asarray(point)
    q1 = 3 * (x_value + 3)**2 + (y_value + 3)**2
    q2 = 3 * (x_value - 3)**2 + (y_value - 3)**2
    e1 = np.exp(-0.05 * q1)
    e2 = np.exp(-0.1 * q2)

    derivative_x = 1.6 * 0.05 * 6 * (x_value + 3) * e1 + 0.1 * 6 * (x_value - 3) * e2
    derivative_y = 1.6 * 0.05 * 2 * (y_value + 3) * e1 + 0.1 * 2 * (y_value - 3) * e2
    return np.array([derivative_x, derivative_y])


def gradient_descent_backtracking(start, max_iterations=200, tolerance=1e-7):
    point = np.asarray(start, dtype=float)
    history = [point.copy()]

    for _ in range(max_iterations):
        gradient = nonconvex_gradient(point)
        gradient_norm = np.linalg.norm(gradient)
        if gradient_norm < tolerance:
            break

        step = 1.0
        current_value = nonconvex_surface(point)
        while nonconvex_surface(point - step * gradient) > current_value - 1e-4 * step * gradient_norm**2:
            step *= 0.5
            if step < 1e-10:
                break

        point = point - step * gradient
        history.append(point.copy())

    return np.array(history)


def alternating_descent(start, cycles=5):
    point = np.asarray(start, dtype=float)
    history = [point.copy()]

    for _ in range(cycles):
        x_result = minimize_scalar(
            lambda x_value: nonconvex_surface([x_value, point[1]]),
            bounds=(-6, 6),
            method="bounded",
        )
        point[0] = x_result.x
        history.append(point.copy())

        y_result = minimize_scalar(
            lambda y_value: nonconvex_surface([point[0], y_value]),
            bounds=(-6, 6),
            method="bounded",
        )
        point[1] = y_result.x
        history.append(point.copy())

    return np.array(history)


nonconvex_starts = [np.array([4.0, 0.0]), np.array([0.0, -5.0]), np.array([-5.0, 2.0])]
nonconvex_paths = [gradient_descent_backtracking(start) for start in nonconvex_starts]
coordinate_path = alternating_descent([4.0, 0.0])

surface_grid = np.linspace(-6, 6, 350)
X_surface, Y_surface = np.meshgrid(surface_grid, surface_grid)
Z_surface = nonconvex_surface([X_surface, Y_surface])

plt.figure(figsize=(10, 7))
plt.contour(X_surface, Y_surface, Z_surface, levels=25, cmap="gray")

for path, color, start in zip(nonconvex_paths, ["tab:red", "tab:purple", "tab:blue"], nonconvex_starts):
    plt.plot(path[:, 0], path[:, 1], "o-", color=color, markersize=3, label=f"GD start {start.tolist()}")

plt.plot(
    coordinate_path[:, 0],
    coordinate_path[:, 1],
    "s--",
    color="tab:green",
    markersize=4,
    label="Alternating descent",
)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Initialization on a Nonconvex Objective")
plt.legend(fontsize=9)
plt.show()

# 4.3 Overdetermined and Underdetermined Systems

For

$$
y=X\beta,
$$

let $n$ be the number of observations and $p$ the number of parameters.

## Overdetermined: $n>p$

There are more equations than unknowns. Noise usually prevents an exact solution, so least squares finds

$$
\hat\beta=\arg\min_\beta\|y-X\beta\|_2^2.
$$

## Underdetermined: $p>n$

There are more unknowns than equations, so many coefficient vectors may satisfy the observations. The pseudoinverse chooses the minimum-$L_2$ solution:

$$
\hat\beta_2
=
\arg\min_\beta\|\beta\|_2
\quad\text{subject to}\quad
X\beta=y.
$$

Basis pursuit chooses the minimum-$L_1$ solution:

$$
\hat\beta_1
=
\arg\min_\beta\|\beta\|_1
\quad\text{subject to}\quad
X\beta=y.
$$

When the true coefficients are sparse and the measurement matrix is suitable, the $L_1$ solution may recover that sparse structure.

In [ ]:
def basis_pursuit(A, b):
    '''Solve min ||x||_1 subject to Ax=b using a linear program.'''
    rows, columns = A.shape

    # Optimization variables are z=[x,u], where u bounds |x|.
    objective = np.r_[np.zeros(columns), np.ones(columns)]
    inequality_matrix = np.block([
        [np.eye(columns), -np.eye(columns)],
        [-np.eye(columns), -np.eye(columns)],
    ])
    inequality_rhs = np.zeros(2 * columns)
    equality_matrix = np.c_[A, np.zeros((rows, columns))]
    bounds = [(None, None)] * columns + [(0, None)] * columns

    result = linprog(
        objective,
        A_ub=inequality_matrix,
        b_ub=inequality_rhs,
        A_eq=equality_matrix,
        b_eq=b,
        bounds=bounds,
        method="highs",
    )
    if not result.success:
        raise RuntimeError(result.message)
    return result.x[:columns]


rng_under = np.random.default_rng(SEED)
n_measurements = 24
n_parameters = 70
n_nonzero = 5

A_under = rng_under.normal(size=(n_measurements, n_parameters)) / np.sqrt(n_measurements)
x_under_true = np.zeros(n_parameters)
support = rng_under.choice(n_parameters, size=n_nonzero, replace=False)
x_under_true[support] = rng_under.normal(size=n_nonzero)
b_under = A_under @ x_under_true

x_under_l2 = np.linalg.pinv(A_under) @ b_under
x_under_l1 = basis_pursuit(A_under, b_under)

under_results = pd.DataFrame({
    "Solution": ["True sparse vector", "Minimum L2", "Minimum L1"],
    "Residual norm": [
        np.linalg.norm(A_under @ x_under_true - b_under),
        np.linalg.norm(A_under @ x_under_l2 - b_under),
        np.linalg.norm(A_under @ x_under_l1 - b_under),
    ],
    "Coefficient L1 norm": [
        np.linalg.norm(x_under_true, 1),
        np.linalg.norm(x_under_l2, 1),
        np.linalg.norm(x_under_l1, 1),
    ],
    "Nonzero count": [
        np.count_nonzero(np.abs(x_under_true) > 1e-6),
        np.count_nonzero(np.abs(x_under_l2) > 1e-6),
        np.count_nonzero(np.abs(x_under_l1) > 1e-6),
    ],
})

print(under_results.round(6).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True, sharey=True)

for ax, values, title in zip(
    axes,
    [x_under_true, x_under_l2, x_under_l1],
    ["True Sparse Coefficients", "Minimum-L2 Solution", "Minimum-L1 Solution"],
):
    ax.bar(np.arange(n_parameters), values, color="tab:blue")
    ax.set_ylabel("Value")
    ax.set_title(title)

axes[-1].set_xlabel("Coefficient index")
plt.tight_layout()
plt.show()

## Regularizing an overdetermined system

With noisy observations, regularization can reduce coefficient variance and improve recovery. Ridge shrinks all coefficients, while LASSO can eliminate weak variables entirely.

In [ ]:
rng_over = np.random.default_rng(SEED + 1)
n_observations = 240
n_features = 60

A_over = rng_over.normal(size=(n_observations, n_features))
x_over_true = np.zeros(n_features)
over_support = rng_over.choice(n_features, size=8, replace=False)
x_over_true[over_support] = rng_over.normal(size=8)
b_over = A_over @ x_over_true + 0.8 * rng_over.standard_normal(n_observations)

x_over_ols = np.linalg.lstsq(A_over, b_over, rcond=None)[0]

ridge_cv = RidgeCV(alphas=np.logspace(-4, 3, 50), fit_intercept=False)
ridge_cv.fit(A_over, b_over)
x_over_ridge = ridge_cv.coef_

lasso_cv = LassoCV(
    alphas=np.logspace(-4, 0, 60),
    cv=5,
    fit_intercept=False,
    max_iter=50_000,
    random_state=SEED,
)
lasso_cv.fit(A_over, b_over)
x_over_lasso = lasso_cv.coef_

over_rows = []
for method, coefficients in {
    "OLS": x_over_ols,
    "Ridge": x_over_ridge,
    "LASSO": x_over_lasso,
}.items():
    over_rows.append({
        "Method": method,
        "Coefficient error": np.linalg.norm(coefficients - x_over_true),
        "Training residual": np.linalg.norm(A_over @ coefficients - b_over),
        "Nonzero coefficients": np.count_nonzero(np.abs(coefficients) > 1e-5),
    })

over_results = pd.DataFrame(over_rows)
print(f"Selected ridge alpha: {ridge_cv.alpha_:.6f}")
print(f"Selected LASSO alpha: {lasso_cv.alpha_:.6f}")
print(over_results.round(5).to_string(index=False))

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True, sharey=True)
for ax, values, title in zip(
    axes,
    [x_over_true, x_over_ols, x_over_ridge, x_over_lasso],
    ["True", "OLS", "RidgeCV", "LassoCV"],
):
    ax.bar(np.arange(n_features), values)
    ax.set_title(title)
    ax.set_ylabel("Value")

axes[-1].set_xlabel("Coefficient index")
plt.tight_layout()
plt.show()

## Matrix-valued regression

The response may contain several outputs at once:

$$
B=AX+E.
$$

Each column of $B$ is a response and each column of $X$ contains its regression coefficients. The same least-squares and regularization ideas apply to all outputs simultaneously.

In [ ]:
rng_matrix = np.random.default_rng(SEED + 2)
n_matrix_observations = 140
n_matrix_features = 35
n_outputs = 7

A_matrix = rng_matrix.normal(size=(n_matrix_observations, n_matrix_features))
X_matrix_true = np.zeros((n_matrix_features, n_outputs))
active_rows = rng_matrix.choice(n_matrix_features, size=8, replace=False)
X_matrix_true[active_rows] = rng_matrix.normal(size=(8, n_outputs))
B_matrix = A_matrix @ X_matrix_true + 0.7 * rng_matrix.normal(
    size=(n_matrix_observations, n_outputs)
)

X_matrix_ols = np.linalg.lstsq(A_matrix, B_matrix, rcond=None)[0]
matrix_ridge = Ridge(alpha=5.0, fit_intercept=False).fit(A_matrix, B_matrix)
X_matrix_ridge = matrix_ridge.coef_.T

fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharex=True, sharey=True)
vmax = np.max(np.abs(X_matrix_true))

for ax, matrix, title in zip(
    axes,
    [X_matrix_true, X_matrix_ols, X_matrix_ridge],
    ["True coefficient matrix", "OLS estimate", "Ridge estimate"],
):
    image = ax.imshow(matrix, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel("Output")
    ax.set_ylabel("Feature")

fig.colorbar(image, ax=axes, shrink=0.75, label="Coefficient value")
plt.show()

# 4.4 Optimization for Regression

Regularized regression solves

$$
\hat\beta
=
\arg\min_\beta
\left[
\|y-X\beta\|_2^2+\lambda R(\beta)
\right].
$$

The official comparison considers several regression strategies:

| Method | Loss and penalty | Effect on coefficients | Main strength | Main limitation |
|---|---|---|---|---|
| **Least squares / OLS** | Squared-error loss; no penalty | Leaves coefficients unrestricted | Simple baseline with the smallest in-sample squared error | Can overfit and become unstable when predictors are numerous or highly correlated |
| **Ridge** | Squared-error loss $+\lambda\|\beta\|_2^2$ | Shrinks coefficients toward zero, but rarely makes them exactly zero | Stabilizes estimates and handles groups of correlated predictors well | Usually retains every predictor, so it does not produce a sparse model |
| **LASSO** | Squared-error loss $+\lambda\|\beta\|_1$ | Shrinks coefficients and can set some exactly to zero | Performs automatic variable selection and produces sparse, interpretable models | May select only one predictor from a group of highly correlated predictors |
| **Elastic Net** | Squared-error loss $+\lambda[\alpha\|\beta\|_1+(1-\alpha)\|\beta\|_2^2]$ | Combines exact zeros from $L_1$ with grouped shrinkage from $L_2$ | Balances sparsity with stability when predictors are correlated | Requires tuning both the total penalty $\lambda$ and the mixing parameter $\alpha$ |
| **Huber regression** | Quadratic loss for small residuals and linear loss for large residuals, often with an $L_2$ penalty | Reduces the influence of observations with unusually large residuals | More robust to outliers than ordinary least squares | Does not inherently create a sparse model and requires choosing the outlier threshold |

Here $\lambda$ controls overall regularization strength. For Elastic Net, $\alpha=1$ gives LASSO-like behavior, while $\alpha=0$ gives Ridge-like behavior.

The next experiment repeatedly fits an overparameterized polynomial library to noisy data. It compares both coefficient stability and clean out-of-sample error.

In [ ]:
rng_compare = np.random.default_rng(SEED + 3)
n_compare = 100
library_degree = 10
n_trials = 60

x_compare = np.linspace(-1, 1, n_compare)
x_compare_test = np.linspace(-1, 1, 400)
Phi_compare = polynomial_matrix(x_compare, library_degree)
Phi_compare_test = polynomial_matrix(x_compare_test, library_degree)


def true_compare_function(x):
    return 0.8 - 1.4 * x + 2.2 * x**2


y_compare_test_true = true_compare_function(x_compare_test)

comparison_models = {
    "Least squares": None,
    "Ridge": Ridge(alpha=0.3, fit_intercept=False),
    "LASSO": Lasso(alpha=0.004, fit_intercept=False, max_iter=50_000),
    "Elastic Net": ElasticNet(
        alpha=0.004,
        l1_ratio=0.5,
        fit_intercept=False,
        max_iter=50_000,
    ),
    "Huber": HuberRegressor(
        epsilon=1.35,
        alpha=0.001,
        fit_intercept=False,
        max_iter=2_000,
    ),
}

comparison_coefficients = {
    name: np.zeros((n_trials, library_degree + 1))
    for name in comparison_models
}
comparison_errors = {name: np.zeros(n_trials) for name in comparison_models}

In [ ]:
for trial in range(n_trials):
    y_training = true_compare_function(x_compare) + 0.18 * rng_compare.standard_normal(n_compare)

    # Add a few large errors so the robust-loss comparison is visible.
    outlier_indices = rng_compare.choice(n_compare, size=4, replace=False)
    y_training[outlier_indices] += rng_compare.normal(0, 2.0, size=4)

    for model_name, model in comparison_models.items():
        if model is None:
            coefficients = np.linalg.lstsq(Phi_compare, y_training, rcond=None)[0]
        else:
            fitted_model = clone(model).fit(Phi_compare, y_training)
            coefficients = fitted_model.coef_

        comparison_coefficients[model_name][trial] = coefficients
        prediction = Phi_compare_test @ coefficients
        comparison_errors[model_name][trial] = rmse(y_compare_test_true, prediction)

comparison_summary = pd.DataFrame({
    "Method": list(comparison_models),
    "Mean test RMSE": [comparison_errors[name].mean() for name in comparison_models],
    "Median test RMSE": [np.median(comparison_errors[name]) for name in comparison_models],
    "Mean coefficient norm": [
        np.mean(np.linalg.norm(comparison_coefficients[name], axis=1))
        for name in comparison_models
    ],
}).sort_values("Mean test RMSE")

print(comparison_summary.round(4).to_string(index=False))

In [ ]:
method_names = list(comparison_models)
fig, axes = plt.subplots(3, 2, figsize=(15, 13))
axes = axes.ravel()

for ax, method_name in zip(axes[:5], method_names):
    ax.boxplot(comparison_coefficients[method_name], showfliers=False)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(method_name)
    ax.set_xlabel("Polynomial coefficient")
    ax.set_ylabel("Estimated value")

axes[5].boxplot(
    [comparison_errors[name] for name in method_names],
    tick_labels=method_names,
    showfliers=False,
)
axes[5].set_title("Clean Test Error")
axes[5].set_ylabel("RMSE")
axes[5].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

# 4.5 The Pareto Front and Parsimonious Models

Model selection balances two competing objectives:

$$
J_1(\beta)=\text{prediction error},
\qquad
J_2(\beta)=\text{model complexity}.
$$

A model is **Pareto optimal** if one objective cannot be improved without worsening the other. The Pareto front contains the best attainable error-complexity combinations.

The regularization parameter moves the model along this tradeoff:

- Small $\lambda$: better training fit, greater complexity
- Large $\lambda$: simpler model, greater training error

The following LASSO path uses the $L_1$ coefficient norm and the number of nonzero coefficients as measures of complexity.

In [ ]:
rng_pareto = np.random.default_rng(SEED + 4)
x_pareto = np.linspace(-1, 1, 140)
y_pareto_true = 1.0 - 1.8 * x_pareto + 0.7 * x_pareto**2 - 0.9 * x_pareto**5
y_pareto = y_pareto_true + 0.18 * rng_pareto.standard_normal(x_pareto.size)

pareto_degree = 15
pareto_features = PolynomialFeatures(degree=pareto_degree, include_bias=False)
X_pareto = pareto_features.fit_transform(x_pareto[:, None])
pareto_scaler = StandardScaler().fit(X_pareto)
X_pareto_scaled = pareto_scaler.transform(X_pareto)

pareto_alphas = np.logspace(-4, 0, 70)
pareto_rows = []

for alpha in pareto_alphas:
    model = Lasso(alpha=alpha, max_iter=100_000).fit(X_pareto_scaled, y_pareto)
    prediction = model.predict(X_pareto_scaled)
    pareto_rows.append({
        "Alpha": alpha,
        "Training RMSE": rmse(y_pareto, prediction),
        "L1 coefficient norm": np.linalg.norm(model.coef_, 1),
        "Nonzero coefficients": np.count_nonzero(np.abs(model.coef_) > 1e-8),
    })

pareto_df = pd.DataFrame(pareto_rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

scatter = axes[0].scatter(
    pareto_df["L1 coefficient norm"],
    pareto_df["Training RMSE"],
    c=np.log10(pareto_df["Alpha"]),
    cmap="viridis",
    s=42,
)
axes[0].set_xlabel(r"Complexity: $\|\beta\|_1$")
axes[0].set_ylabel("Training RMSE")
axes[0].set_title("LASSO Pareto Front")
fig.colorbar(scatter, ax=axes[0], label=r"$\log_{10}(\alpha)$")

axes[1].plot(
    pareto_df["Nonzero coefficients"],
    pareto_df["Training RMSE"],
    "o-",
    color="tab:purple",
    markersize=4,
)
axes[1].set_xlabel("Number of nonzero coefficients")
axes[1].set_ylabel("Training RMSE")
axes[1].set_title("Fit Versus Selected Model Size")

plt.tight_layout()
plt.show()

# 4.6 Model Selection and Cross-Validation

## Holdout validation

The simplest validation method fits a model on one subset and evaluates it on another:

$$
\text{training data}\rightarrow\hat\beta
\rightarrow\text{validation loss}.
$$

The next example fits polynomials on $x\in[0,4]$ and evaluates extrapolation on $x\in(4,8]$. Increasing the degree can improve in-sample fit while making extrapolation unstable.

In [ ]:
def scaled_polynomial_matrix(x, degree, center, scale):
    normalized_x = (np.asarray(x) - center) / scale
    return polynomial_matrix(normalized_x, degree)


rng_holdout = np.random.default_rng(SEED + 5)
x_holdout = np.linspace(0, 8, 200)
y_holdout_true = x_holdout**2
y_holdout = y_holdout_true + 0.7 * rng_holdout.standard_normal(x_holdout.size)

train_mask = x_holdout <= 4
x_holdout_train = x_holdout[train_mask]
y_holdout_train = y_holdout[train_mask]
x_holdout_validation = x_holdout[~train_mask]
y_holdout_validation = y_holdout[~train_mask]

holdout_center = x_holdout_train.mean()
holdout_scale = (x_holdout_train.max() - x_holdout_train.min()) / 2
holdout_degrees = np.arange(0, 16)
holdout_train_rmse = []
holdout_validation_rmse = []
holdout_coefficients = {}

for degree in holdout_degrees:
    X_train_degree = scaled_polynomial_matrix(
        x_holdout_train, degree, holdout_center, holdout_scale
    )
    X_validation_degree = scaled_polynomial_matrix(
        x_holdout_validation, degree, holdout_center, holdout_scale
    )
    coefficients = np.linalg.lstsq(X_train_degree, y_holdout_train, rcond=None)[0]

    holdout_coefficients[degree] = coefficients
    holdout_train_rmse.append(rmse(y_holdout_train, X_train_degree @ coefficients))
    holdout_validation_rmse.append(
        rmse(y_holdout_validation, X_validation_degree @ coefficients)
    )

holdout_train_rmse = np.array(holdout_train_rmse)
holdout_validation_rmse = np.array(holdout_validation_rmse)
best_holdout_degree = holdout_degrees[np.argmin(holdout_validation_rmse)]

print("Degree selected by holdout validation:", best_holdout_degree)
print(f"Minimum validation RMSE: {holdout_validation_rmse.min():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

axes[0].semilogy(holdout_degrees, holdout_train_rmse, "o-", label="Training RMSE")
axes[0].semilogy(holdout_degrees, holdout_validation_rmse, "o-", label="Validation RMSE")
axes[0].axvline(best_holdout_degree, color="black", linestyle="--", label="Selected degree")
axes[0].set_xlabel("Polynomial degree")
axes[0].set_ylabel("RMSE (log scale)")
axes[0].set_title("Interpolation Versus Extrapolation Error")
axes[0].legend()

axes[1].scatter(x_holdout_train, y_holdout_train, s=14, color="tab:blue", alpha=0.55, label="Training")
axes[1].scatter(
    x_holdout_validation,
    y_holdout_validation,
    s=14,
    color="tab:orange",
    alpha=0.55,
    label="Validation",
)
axes[1].plot(x_holdout, y_holdout_true, color="black", linewidth=2.5, label="Underlying function")

for degree, color in zip([1, 2, 5, 12], ["tab:green", "tab:red", "tab:purple", "tab:brown"]):
    X_full_degree = scaled_polynomial_matrix(
        x_holdout, degree, holdout_center, holdout_scale
    )
    axes[1].plot(
        x_holdout,
        X_full_degree @ holdout_coefficients[degree],
        color=color,
        linewidth=1.6,
        label=f"Degree {degree}",
    )

axes[1].axvline(4, color="gray", linestyle="--")
axes[1].set_ylim(-10, 90)
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("Fits Trained Only on the Left Half")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## $K$-fold cross-validation

In $K$-fold cross-validation, each fold serves once as validation data while the remaining folds are used for training:

$$
CV(d)=\frac{1}{K}\sum_{k=1}^{K}L_{k}^{\text{validation}}(d),
$$

where $d$ may represent polynomial degree or another hyperparameter.

The final test set is not used during this selection. After choosing the degree, the model is refitted on all available training data and evaluated once on the test set.

In [ ]:
rng_cv = np.random.default_rng(SEED + 6)
x_cv = rng_cv.uniform(-2, 2, size=260)
y_cv_true = 1.0 - 0.7 * x_cv + 1.4 * x_cv**2 - 0.55 * x_cv**3
y_cv = y_cv_true + 0.9 * rng_cv.standard_normal(x_cv.size)

x_cv_train, x_cv_test, y_cv_train, y_cv_test = train_test_split(
    x_cv,
    y_cv,
    test_size=0.2,
    random_state=SEED,
)

cv_degrees = np.arange(1, 13)
kfold = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []

for degree in cv_degrees:
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        StandardScaler(),
        Ridge(alpha=1e-5),
    )
    scores = cross_validate(
        model,
        x_cv_train[:, None],
        y_cv_train,
        cv=kfold,
        scoring="neg_mean_squared_error",
        return_train_score=True,
    )
    fold_training_rmse = np.sqrt(-scores["train_score"])
    fold_validation_rmse = np.sqrt(-scores["test_score"])
    cv_rows.append({
        "Degree": degree,
        "Training RMSE": fold_training_rmse.mean(),
        "Validation RMSE": fold_validation_rmse.mean(),
        "Validation SE": fold_validation_rmse.std(ddof=1)
        / np.sqrt(kfold.get_n_splits()),
    })

kfold_results = pd.DataFrame(cv_rows)
best_cv_degree = int(
    kfold_results.loc[kfold_results["Validation RMSE"].idxmin(), "Degree"]
)

final_cv_model = make_pipeline(
    PolynomialFeatures(degree=best_cv_degree, include_bias=False),
    StandardScaler(),
    Ridge(alpha=1e-5),
)
final_cv_model.fit(x_cv_train[:, None], y_cv_train)
test_prediction = final_cv_model.predict(x_cv_test[:, None])
final_test_rmse = rmse(y_cv_test, test_prediction)

print(kfold_results.round(4).to_string(index=False))
print(f"\nSelected degree: {best_cv_degree}")
print(f"Untouched test RMSE: {final_test_rmse:.4f}")

In [ ]:
plt.figure(figsize=(10, 5.5))
plt.plot(
    kfold_results["Degree"],
    kfold_results["Training RMSE"],
    "o-",
    label="Mean training RMSE",
)
plt.plot(
    kfold_results["Degree"],
    kfold_results["Validation RMSE"],
    "o-",
    label="Mean validation RMSE",
)
plt.axvline(best_cv_degree, color="black", linestyle="--", label="Selected degree")
plt.xlabel("Polynomial degree")
plt.ylabel("RMSE")
plt.title("Five-Fold Cross-Validation")
plt.legend()
plt.show()

## Cross-validation for time-series data

Random folds are inappropriate when observations are time ordered because they can train on the future and validate on the past. `TimeSeriesSplit` uses expanding training windows and later validation blocks, preserving chronology.

In [ ]:
time_indices = np.arange(60)
time_series_split = TimeSeriesSplit(n_splits=5)

plt.figure(figsize=(12, 4.5))

for fold, (train_indices, validation_indices) in enumerate(
    time_series_split.split(time_indices), start=1
):
    plt.scatter(train_indices, np.full_like(train_indices, fold), marker="s", s=35, color="tab:blue")
    plt.scatter(
        validation_indices,
        np.full_like(validation_indices, fold),
        marker="s",
        s=35,
        color="tab:orange",
    )

plt.xlabel("Time index")
plt.ylabel("Fold")
plt.title("Expanding-Window Time-Series Cross-Validation")
plt.yticks(range(1, 6))
plt.scatter([], [], marker="s", color="tab:blue", label="Training")
plt.scatter([], [], marker="s", color="tab:orange", label="Validation")
plt.legend()
plt.show()

# 4.7 Model Selection and Information Criteria

## KL divergence

The Kullback–Leibler divergence measures how one probability distribution differs from another:

$$
D_{KL}(f\|g)
=
\int f(x)\log\left(\frac{f(x)}{g(x)}\right)\,dx.
$$

It satisfies $D_{KL}(f\|g)\geq0$ and equals zero only when the distributions agree almost everywhere. It is not symmetric, so it is not a distance metric in the usual geometric sense.

The official validation example compares a reference density with several candidate models.

In [ ]:
def normalize_density(density, grid):
    return density / np.trapezoid(density, grid)


def kl_divergence(reference_density, candidate_density, grid, epsilon=1e-12):
    reference = np.clip(reference_density, epsilon, None)
    candidate = np.clip(candidate_density, epsilon, None)
    return np.trapezoid(reference * np.log(reference / candidate), grid)


x_density = np.linspace(-6, 6, 2401)
reference_density = normalize_density(norm.pdf(x_density, loc=0.0, scale=1.0), x_density)
candidate_1 = normalize_density(norm.pdf(x_density, loc=0.7, scale=0.9), x_density)
candidate_2 = normalize_density(
    0.75 * norm.pdf(x_density, loc=0.0, scale=1.0)
    + 0.25 * norm.pdf(x_density, loc=-3.0, scale=0.7),
    x_density,
)
candidate_3 = normalize_density(
    np.where((x_density >= -0.5) & (x_density <= 4.5), 1.0, 1e-8),
    x_density,
)

density_candidates = {
    "Shifted Gaussian": candidate_1,
    "Gaussian mixture": candidate_2,
    "Approx. uniform": candidate_3,
}

kl_rows = []
for name, candidate in density_candidates.items():
    kl_rows.append({
        "Candidate": name,
        "D_KL(reference || candidate)": kl_divergence(
            reference_density, candidate, x_density
        ),
    })

kl_results = pd.DataFrame(kl_rows).sort_values("D_KL(reference || candidate)")
print(kl_results.round(5).to_string(index=False))

plt.figure(figsize=(11, 5.5))
plt.plot(x_density, reference_density, color="black", linewidth=3, label="Reference")
for name, candidate in density_candidates.items():
    plt.plot(x_density, candidate, linewidth=1.8, label=name)
plt.xlabel("x")
plt.ylabel("Probability density")
plt.title("Reference Distribution and Candidate Models")
plt.legend()
plt.show()

## AIC and BIC

Information criteria combine maximum likelihood with a penalty for model size:

$$
AIC=2k-2\log(\hat L),
$$

$$
BIC=k\log(n)-2\log(\hat L),
$$

where $k$ is the number of estimated parameters, $n$ is the sample size, and $\hat L$ is the maximized likelihood.

Smaller values are preferred. AIC emphasizes predictive accuracy, while BIC generally penalizes additional parameters more strongly as $n$ grows.

The original companion example uses a now-removed `statsmodels` ARMA interface. The modernized version below performs the same AR-order comparison directly with least squares and a Gaussian residual likelihood.

In [ ]:
rng_information = np.random.default_rng(123)
n_ar_total = 700
burn_in = 100
ar_series = np.zeros(n_ar_total)
innovations = rng_information.normal(scale=1.0, size=n_ar_total)

# True AR(2) data-generating process.
for time_index in range(2, n_ar_total):
    ar_series[time_index] = (
        0.65 * ar_series[time_index - 1]
        - 0.30 * ar_series[time_index - 2]
        + innovations[time_index]
    )

ar_series = ar_series[burn_in:]
max_ar_order = 8
common_start = max_ar_order
ar_target = ar_series[common_start:]
information_rows = []

for order in range(max_ar_order + 1):
    design_columns = [np.ones(ar_target.size)]
    for lag in range(1, order + 1):
        design_columns.append(
            ar_series[common_start - lag : ar_series.size - lag]
        )
    design_matrix = np.column_stack(design_columns)

    coefficients = np.linalg.lstsq(design_matrix, ar_target, rcond=None)[0]
    residuals = ar_target - design_matrix @ coefficients
    residual_sum_squares = residuals @ residuals
    n_effective = ar_target.size
    variance_mle = residual_sum_squares / n_effective

    log_likelihood = -0.5 * n_effective * (
        np.log(2 * np.pi) + 1 + np.log(variance_mle)
    )

    # AR coefficients + intercept + residual variance.
    parameter_count = order + 2
    aic_value = 2 * parameter_count - 2 * log_likelihood
    bic_value = parameter_count * np.log(n_effective) - 2 * log_likelihood

    information_rows.append({
        "AR order": order,
        "Log likelihood": log_likelihood,
        "AIC": aic_value,
        "BIC": bic_value,
    })

information_results = pd.DataFrame(information_rows)
best_aic_order = int(information_results.loc[information_results["AIC"].idxmin(), "AR order"])
best_bic_order = int(information_results.loc[information_results["BIC"].idxmin(), "AR order"])

print(information_results.round(3).to_string(index=False))
print(f"\nAIC-selected order: {best_aic_order}")
print(f"BIC-selected order: {best_bic_order}")

In [ ]:
plt.figure(figsize=(10, 5.5))
plt.plot(information_results["AR order"], information_results["AIC"], "o-", label="AIC")
plt.plot(information_results["AR order"], information_results["BIC"], "s-", label="BIC")
plt.axvline(2, color="black", linestyle="--", label="True AR order")
plt.xlabel("Candidate AR order")
plt.ylabel("Information criterion")
plt.title("AIC and BIC Model Selection")
plt.xticks(information_results["AR order"])
plt.legend()
plt.show()

# Chapter 4 Summary

- Regression estimates parameters by minimizing a loss function.
- The choice of loss changes sensitivity to large residuals and outliers.
- Linear-in-parameter models can be solved with least squares even when their features are nonlinear in the input.
- Gradient descent is needed when a direct solution is unavailable, but nonconvex objectives may contain local minima.
- Overdetermined systems require approximation; underdetermined systems require an additional selection principle.
- Regularization trades training fit for stability, sparsity, or simplicity.
- Pareto fronts visualize the competition between error and model complexity.
- Cross-validation selects hyperparameters using held-out data.
- Chronological validation is required for time series.
- KL divergence compares probability distributions, while AIC and BIC compare likelihood-based models with complexity penalties.

The central principle is

$$
\boxed{
\text{Choose the simplest model that captures the important structure and generalizes well.}
}
$$